# IMERG GPM Final Run — Download, Mapping, and Basin-Averaged Precipitation

This notebook provides a step-by-step workflow for:

1. **Downloading** (optional) IMERG GPM *Final Run* precipitation data at 30-minute, daily, or monthly temporal resolution over a user-defined period and area. The area of interest can be selected interactively by drawing a rectangle on a map. Alternatively, the notebook can work directly with previously downloaded files.
2. Creating **precipitation maps** (optional) showing the mean, maximum, and minimum precipitation over the selected area, with optional export to Excel.
3. **Loading a basin shapefile**, checking its geometry validity, calculating its area, comparing it with the IMERG grid, and producing maps focused on the basin.
4. Computing **basin-averaged precipitation** using three different spatial aggregation methods:

   * grid-cell centroid selection;
   * raster masking with `all_touched=True` or `all_touched=False`;
   * area-weighted Voronoi polygons.

   The resulting time series can be compared both directly and in terms of cumulative precipitation. A specific method can then be selected for the final data export.

## Folder structure

The notebook assumes the following folder structure, starting from the directory containing the notebook itself:

```text
notebook_folder/
├── IMERG_notebook.ipynb
├── .netrc                 <- NASA Earthdata credentials (see Section 0)
├── shapefile/             <- basin shapefile (.shp, .shx, .dbf, .prj, ...)
├── nc/                    <- downloaded IMERG files (created automatically)
└── output/                <- maps, plots, and Excel files (created automatically)
```

> Downloading IMERG data requires a free NASA Earthdata account: https://urs.earthdata.gov/users/new



## 0. Environment setup

In [ ]:
# Package installation (run only once)

In [ ]:
# ============================================================
# 1. Standard Python libraries
# ============================================================
# File and folder management, text processing, dates, and warnings

import os
import re
import glob
import datetime as dt
import warnings

# Hide non-critical warning messages to keep the notebook output clean
warnings.filterwarnings("ignore")


# ============================================================
# 2. Data handling and numerical analysis
# ============================================================
# Work with arrays, tables, and multidimensional datasets

import numpy as np
import pandas as pd
import xarray as xr
import netCDF4


# ============================================================
# 3. Geographic and vector data
# ============================================================
# Read and manipulate shapefiles and geometric objects

import geopandas as gpd
import shapely.geometry as sgeom
from shapely.geometry import box, Point


# ============================================================
# 4. Raster data
# ============================================================
# Read, create, and manipulate raster datasets

import rasterio
from rasterio import features
from rasterio.transform import from_origin


# ============================================================
# 5. Plotting and map visualization
# ============================================================
# Create plots and geographic maps

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import matplotlib.dates as mdates

import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt


# ============================================================
# 6. Interactive notebook tools
# ============================================================
# Create interactive controls and display objects in Jupyter

import ipywidgets as widgets
from IPython.display import display

from ipyleaflet import Map, DrawControl, basemaps, GeoJSON


# ============================================================
# 7. NASA Earthdata access
# ============================================================
# Authenticate and download datasets from NASA Earthdata

import earthaccess
import netrc


### Working directories

The notebook uses and create three folders located relative to the notebook directory:

In [ ]:
# ============================================================
# Folder structure
# ============================================================
# Define the folders used to store input data and notebook outputs

NC_DIR = "nc"            # Folder where IMERG GPM NetCDF files will be stored
OUT_DIR = "output"       # Folder where processed results will be saved
SHP_DIR = "shapefile"    # Folder containing the catchment shapefile


# Create the input/output folders if they do not already exist
os.makedirs(NC_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)


# Check that the shapefile folder exists before continuing
assert os.path.isdir(SHP_DIR), (
    f"The folder '{SHP_DIR}/' was not found. "
    "Please create it and place the catchment shapefile inside."
)

### NASA Earthdata login
The notebook expects a .netrc file containing the NASA Earthdata credentials to be located in the same directory as the notebook.

In this workflow, the .netrc file is intentionally read from the notebook folder rather than from the user's home directory.

The file must have the following format:

```
machine urs.earthdata.gov
login IL_TUO_USERNAME
password LA_TUA_PASSWORD
```

Replace YOUR_USERNAME and YOUR_PASSWORD with the credentials associated with your NASA Earthdata account.




In [ ]:
# ============================================================
# NASA Earthdata authentication
# ============================================================
# Read NASA Earthdata credentials from the local .netrc file and use them to authenticate through the earthaccess library

NETRC_PATH = os.path.join(os.getcwd(), ".netrc")


# Check that the .netrc file exists in the current working directory
assert os.path.isfile(NETRC_PATH), (
    f".netrc file not found at: {NETRC_PATH}\n"
    "Please create the file before continuing."
)


# Read the Earthdata username and password from .netrc
_netrc = netrc.netrc(NETRC_PATH)
_username, _, _password = _netrc.authenticators("urs.earthdata.gov")


# Make the credentials available to earthaccess
os.environ["EARTHDATA_USERNAME"] = _username
os.environ["EARTHDATA_PASSWORD"] = _password


# Log in to NASA Earthdata
auth = earthaccess.login(strategy="environment")


# Check whether the authentication was successful
if auth.authenticated:
    print("NASA Earthdata login successful.")
else:
    print("NASA Earthdata login failed.")

## 1. Interactive selection: temporal resolution, period, area, and options

In this section, the user can interactively define the main settings used throughout the notebook:

- **Temporal resolution**: select 30-minute, daily, or monthly (*IMERG GPM Final Run*).
- **Time period**: specify the start and end dates of the analysis.
- **Area of interest**: draw a rectangle on the interactive map to define the spatial extent.
- **Data availability**: the notebook automatically checks whether locally available IMERG files cover the selected period and area. If the existing files are sufficient, they are used directly. Otherwise, the required IMERG data are automatically downloaded from NASA Earthdata.
- **Processing options**: enable or disable data download, figure saving, spatial precipitation maps, and Excel export for the selected area.

### Notebook execution options
The following variables control which optional parts of the workflow are executed.

* **`save_plots`**
  If `True`, generated figures are also saved in the `output/` folder.
  If `False`, figures are displayed only in the notebook.

* **`run_area_maps`**
  If `True`, spatial maps of mean, maximum, and minimum precipitation are generated for the selected area.

* **`run_area_excel`**
  If `True`, spatial precipitation statistics for the selected area are exported to Excel.



In [ ]:
# ============================================================
# Analysis settings
# ============================================================
# Use the controls below to select the IMERG product, the analysis period, and the outputs to generate.

# ------------------------------------------------------------
# 1. IMERG product and temporal resolution
# ------------------------------------------------------------
# Each temporal resolution corresponds to a different NASA GPM IMERG product.

IMERG_PRODUCTS = {
    "30 min": {"short_name": "GPM_3IMERGHH", "version": "07"},
    "Daily": {"short_name": "GPM_3IMERGDF", "version": "07"},
    "Monthly": {"short_name": "GPM_3IMERGM", "version": "07"},
}

time_scale_w = widgets.Dropdown(
    options=list(IMERG_PRODUCTS.keys()),
    value="Daily",
    description="Temporal resolution:",
    style={"description_width": "initial"},
)


# ------------------------------------------------------------
# 2. Analysis period
# ------------------------------------------------------------
# Select the start and end dates of the period to analyse.

start_date_w = widgets.DatePicker(
    description="Start date:",
    value=dt.date(2023, 6, 1),
)

end_date_w = widgets.DatePicker(
    description="End date:",
    value=dt.date(2023, 6, 30),
)


# ------------------------------------------------------------
# 3. Processing and output options
# ------------------------------------------------------------
# Select which additional outputs should be generated.

save_plots_w = widgets.Checkbox(
    value=True,
    description="Save figures",
)

run_area_maps_w = widgets.Checkbox(
    value=True,
    description="Generate mean, maximum, and minimum precipitation maps",
)

run_area_excel_w = widgets.Checkbox(
    value=True,
    description="Export precipitation statistics to Excel",
)


# ------------------------------------------------------------
# Display the interactive controls
# ------------------------------------------------------------

display(time_scale_w)

display(
    widgets.HBox([
        start_date_w,
        end_date_w,
    ])
)

display(
    widgets.VBox([
        save_plots_w,
        run_area_maps_w,
        run_area_excel_w,
    ])
)


### Figure-saving utility

The following helper function is used throughout the notebook to save figures only when `save_plots=True`.

In [ ]:
# ============================================================
# Helper function for saving figures
# ============================================================

def maybe_save(fig, filename):
    """
    Save a figure to the output folder if figure saving is enabled.

    """

    # Build the complete output path
    path = os.path.join(OUT_DIR, filename)

    # Save the figure only if this option was selected
    if save_plots:
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print(f"Figure saved to: {path}")
    else:
        print("Figure saving is disabled.")

In [ ]:
# ============================================================
# Interactive area selection
# ============================================================
# Draw a rectangle on the map to define the geographic area
# used for the IMERG data search and download.


# The selected bounding box will be stored as:
# (longitude_min, latitude_min, longitude_max, latitude_max)
selected_bbox = {"bbox": None}


# Create an interactive map centered over Italy
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(43.1, 12.4),
    zoom=6,
)


# Add a drawing tool.
# Only rectangular selections are enabled.
draw_control = DrawControl(
    rectangle={
        "shapeOptions": {
            "color": "#e63946",
            "fillOpacity": 0.2,
        }
    },
    polygon={},
    circle={},
    circlemarker={},
    polyline={},
    marker={},
)


def handle_draw(target, action, geo_json):
    """
    Extract the bounding box coordinates from the rectangle
    drawn on the interactive map.
    """

    # Process newly created or edited rectangles
    if action in ("created", "edited") and geo_json["geometry"]["type"] == "Polygon":

        # Extract the rectangle coordinates
        coords = geo_json["geometry"]["coordinates"][0]

        lons = [coord[0] for coord in coords]
        lats = [coord[1] for coord in coords]

        # Store the bounding box
        selected_bbox["bbox"] = (
            min(lons),
            min(lats),
            max(lons),
            max(lats),
        )

        print(
            "Selected area "
            "(lon_min, lat_min, lon_max, lat_max):",
            selected_bbox["bbox"],
        )


# Connect the drawing tool to the function above
draw_control.on_draw(handle_draw)

# Add the drawing tool to the map
m.add(draw_control)

# Display the interactive map
m


In [ ]:
# ============================================================
# Read and validate the selected settings
# ============================================================
# Run this cell only AFTER drawing the study area on the map.


# ------------------------------------------------------------
# 1. Geographic area
# ------------------------------------------------------------

# Check that a bounding box has been selected
assert selected_bbox["bbox"] is not None, (
    "Draw a rectangle on the map before continuing."
)

bbox = selected_bbox["bbox"]


# ------------------------------------------------------------
# 2. Temporal settings
# ------------------------------------------------------------

# Read the selected temporal resolution and analysis period
time_scale = time_scale_w.value
start_date = start_date_w.value
end_date = end_date_w.value


# Check that both dates have been selected
assert start_date is not None and end_date is not None, (
    "Please select both a start date and an end date."
)


# Check that the selected period is valid
assert start_date <= end_date, (
    "The start date must be earlier than or equal to the end date."
)


# ------------------------------------------------------------
# 3. Processing options
# ------------------------------------------------------------

# Read the options selected in the interactive controls
save_plots = save_plots_w.value
run_area_maps = run_area_maps_w.value
run_area_excel = run_area_excel_w.value


# ------------------------------------------------------------
# 4. Summary of the selected settings
# ------------------------------------------------------------

print("Selected settings")
print("-----------------")
print(f"Temporal resolution: {time_scale}")
print(f"Period: {start_date} → {end_date}")
print(f"Bounding box (lon_min, lat_min, lon_max, lat_max): {bbox}")
print(f"Save figures: {save_plots}")
print(f"Generate area precipitation maps: {run_area_maps}")
print(f"Export area statistics to Excel: {run_area_excel}")

## 2. IMERG data search and optional download

The notebook first checks whether the required IMERG files are already available locally.

If the local files fully cover the selected analysis period, they are used directly. Otherwise, the missing data are searched and downloaded automatically from NASA Earthdata.

Finally, the notebook checks that the available files provide complete temporal and spatial coverage for the selected analysis.


### Functions used to identify and check IMERG files

In [ ]:
# ============================================================
# 2.1 Functions for IMERG file management
# ============================================================


def list_imerg_files(folder):
    """
    Return all IMERG HDF5 and NetCDF files available in a folder.
    """

    patterns = ["*.HDF5", "*.hdf5", "*.nc4", "*.nc"]
    files = []

    for pattern in patterns:
        files.extend(glob.glob(os.path.join(folder, pattern)))

    return sorted(set(files))


def get_file_timestamp(file_path):
    """
    Extract the acquisition start time from an IMERG filename.

    Example
    -------
    3B-HHR.MS.MRG.3IMERG.20230601-S003000-E005959...
    """

    filename = os.path.basename(file_path)

    match = re.search(
        r"(\d{8})-S(\d{6})-E",
        filename,
    )

    if match is None:
        raise ValueError(
            "Could not determine the acquisition time from "
            f"IMERG filename:\n{filename}"
        )

    date_string = match.group(1)
    time_string = match.group(2)

    return pd.to_datetime(
        date_string + time_string,
        format="%Y%m%d%H%M%S",
    )


def expected_time_keys(start_date, end_date, time_scale):
    """
    Create the expected timesteps for the selected analysis period.
    """

    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    if time_scale == "30 min":

        # Include all 30-minute intervals of the final selected day
        return set(
            pd.date_range(
                start=start,
                end=end + pd.Timedelta(days=1) - pd.Timedelta(minutes=30),
                freq="30min",
            )
        )

    elif time_scale == "Daily":

        return set(
            pd.date_range(
                start=start,
                end=end,
                freq="D",
            )
        )

    elif time_scale == "Monthly":

        return set(
            pd.period_range(
                start=start,
                end=end,
                freq="M",
            )
        )

    else:
        raise ValueError(
            f"Unsupported temporal resolution: {time_scale}"
        )


def file_time_key(file_path, time_scale):
    """
    Return the timestep represented by an IMERG file.
    """

    timestamp = get_file_timestamp(file_path)

    if time_scale == "30 min":
        return timestamp.floor("30min")

    elif time_scale == "Daily":
        return timestamp.normalize()

    elif time_scale == "Monthly":
        return timestamp.to_period("M")

    else:
        raise ValueError(
            f"Unsupported temporal resolution: {time_scale}"
        )


def select_files_for_period(file_list, start_date, end_date, time_scale):
    """
    Select only the IMERG files belonging to the requested period.
    """

    expected = expected_time_keys(
        start_date,
        end_date,
        time_scale,
    )

    selected = []

    for file_path in file_list:

        key = file_time_key(
            file_path,
            time_scale,
        )

        if key in expected:
            selected.append(file_path)

    return sorted(selected)


def check_temporal_coverage(file_list, start_date, end_date, time_scale):
    """
    Check whether all expected timesteps are available locally.
    """

    expected = expected_time_keys(
        start_date,
        end_date,
        time_scale,
    )

    available = {
        file_time_key(file_path, time_scale)
        for file_path in file_list
    }

    missing = expected - available

    return len(missing) == 0, missing


def get_imerg_spatial_extent(file_path, time_scale):
    """
    Read the geographic extent covered by an IMERG file.

    """

    # Daily files have a slightly different internal structure
    if time_scale == "Daily":
        ds = xr.open_dataset(
            file_path,
            engine="netcdf4",
        )
    else:
        ds = xr.open_dataset(
            file_path,
            group="Grid",
            engine="netcdf4",
        )

    try:

        lons = ds["lon"].values
        lats = ds["lat"].values

        # Estimate the grid-cell size
        dlon = (
            np.abs(np.median(np.diff(lons)))
            if len(lons) > 1
            else 0.1
        )

        dlat = (
            np.abs(np.median(np.diff(lats)))
            if len(lats) > 1
            else 0.1
        )

        # Convert grid-cell centres to outer grid boundaries
        lon_min = np.min(lons) - dlon / 2
        lon_max = np.max(lons) + dlon / 2
        lat_min = np.min(lats) - dlat / 2
        lat_max = np.max(lats) + dlat / 2

    finally:
        ds.close()

    return lon_min, lat_min, lon_max, lat_max


def check_spatial_coverage(file_path, requested_bbox, time_scale):
    """
    Check whether an IMERG file covers the selected study area.
    """

    available_bbox = get_imerg_spatial_extent(
        file_path,
        time_scale,
    )

    req_lon_min, req_lat_min, req_lon_max, req_lat_max = requested_bbox
    lon_min, lat_min, lon_max, lat_max = available_bbox

    covered = (
        req_lon_min >= lon_min
        and req_lon_max <= lon_max
        and req_lat_min >= lat_min
        and req_lat_max <= lat_max
    )

    return covered, available_bbox

### Check local data and download if necessary

IMERG files are stored in separate folders according to their temporal resolution.

The notebook checks whether the locally available files cover the selected period and study area. If the local dataset is incomplete, the required IMERG granules are automatically searched and downloaded from NASA Earthdata.

In [ ]:
# ============================================================
# 2.2 Check local data and download if necessary
# ============================================================


# Create a subfolder for the selected temporal resolution
scale_dir = os.path.join(
    NC_DIR,
    time_scale.replace(" ", "_"),
)

os.makedirs(scale_dir, exist_ok=True)


# ------------------------------------------------------------
# Check locally available IMERG files
# ------------------------------------------------------------

local_files = list_imerg_files(scale_dir)

print(f"Local IMERG files found: {len(local_files)}")


# Initial coverage status
temporal_ok = False
spatial_ok = False
missing_times = set()


# Check temporal and spatial coverage when local files exist
if local_files:

    temporal_ok, missing_times = check_temporal_coverage(
        local_files,
        start_date,
        end_date,
        time_scale,
    )

    spatial_ok, available_bbox = check_spatial_coverage(
        local_files[0],
        bbox,
        time_scale,
    )


# ------------------------------------------------------------
# Download data only if local coverage is incomplete
# ------------------------------------------------------------

if temporal_ok and spatial_ok:

    print("Local IMERG data fully cover the selected period and area.")
    print("No download is required.")

else:

    print("Local IMERG data are incomplete.")

    if not temporal_ok:
        print(
            f"Temporal coverage incomplete: "
            f"{len(missing_times)} timestep(s) missing."
        )

    if local_files and not spatial_ok:
        print(
            "Spatial coverage does not include the full selected area."
        )

    if not local_files:
        print(
            "No local IMERG data are available "
            "for this temporal resolution."
        )

    print("Searching NASA Earthdata...")

    # Select the IMERG product corresponding to the chosen resolution
    product = IMERG_PRODUCTS[time_scale]

    temporal_start = f"{start_date.isoformat()}T00:00:00"
    temporal_end = f"{end_date.isoformat()}T23:59:59"

    # Search NASA Earthdata
    results = earthaccess.search_data(
        short_name=product["short_name"],
        version=product["version"],
        temporal=(temporal_start, temporal_end),
        bounding_box=bbox,
    )

    print(f"IMERG granules found: {len(results)}")

    assert len(results) > 0, (
        "No IMERG data were found for the selected period and area."
    )

    # Download the selected granules
    earthaccess.download(
        results,
        scale_dir,
    )

    print(f"IMERG data stored in: {scale_dir}")

### Select and validate the files used in the analysis

After the local check and possible download, only the files belonging to the selected analysis period are retained.

A final check ensures that the dataset contains all expected timesteps and covers the complete study area.

In [ ]:
# ============================================================
# 2.3 Select and validate the IMERG files
# ============================================================


# Read the IMERG files currently available in the folder
local_files = list_imerg_files(scale_dir)


# Select only the files belonging to the requested period
imerg_files = select_files_for_period(
    local_files,
    start_date,
    end_date,
    time_scale,
)


# Check that at least one file is available
assert len(imerg_files) > 0, (
    "No IMERG files are available for the selected period."
)


# ------------------------------------------------------------
# Final temporal coverage check
# ------------------------------------------------------------

temporal_ok, missing_times = check_temporal_coverage(
    imerg_files,
    start_date,
    end_date,
    time_scale,
)

assert temporal_ok, (
    f"IMERG temporal coverage is incomplete. "
    f"{len(missing_times)} timestep(s) are missing."
)


# ------------------------------------------------------------
# Final spatial coverage check
# ------------------------------------------------------------

spatial_ok, available_bbox = check_spatial_coverage(
    imerg_files[0],
    bbox,
    time_scale,
)

assert spatial_ok, (
    "The available IMERG files do not fully cover the selected area."
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(f"Files used for the analysis: {len(imerg_files)}")
print("IMERG temporal coverage: OK")
print("IMERG spatial coverage: OK")

## 3. Loading and cropping IMERG data to the selected area

IMERG data are distributed as **NetCDF/HDF5 files**, a format widely used for scientific and geospatial datasets. Unlike a simple table, a NetCDF file can store several variables together with their dimensions, coordinates, units, and metadata in a structured way.

For example, an IMERG file contains the `precipitation` variable together with the corresponding longitude, latitude, and time coordinates.

NetCDF/HDF5 files can also organize variables into **groups**, which work similarly to folders inside the file. In the IMERG products used here, the internal structure depends on the selected temporal resolution:

- the **30-minute** and **monthly** products store the `precipitation` variable inside a group called **`Grid`**;
- the **daily** product stores the variables directly in the **root group**, with no `Grid` group.

This distinction is important when opening the files with `xarray`. Trying to open a daily file with `group="Grid"` raises the error:

`OSError: [Errno group not found: Grid] 'Grid'`

For this reason, the helper function `get_imerg_group()` automatically checks the internal structure of each file and determines whether the data should be read from the `Grid` group or directly from the root.

The following cell inspects the structure of one IMERG file and displays its groups, dimensions, and variables.

The native precipitation variable is organized with dimensions in the order:

`time, lon, lat`

For easier spatial analysis and visualization, the data are later transposed to:

`time, lat, lon`

This ordering is more convenient for operations such as mapping, clipping, and raster processing.


### Inspect the raw file structure

Before loading the data, it helps to look at the actual structure of one IMERG file: which groups it contains, and which variables/dimensions live in each one. This is also how the `get_imerg_group()` helper below decides whether to open the file with `group="Grid"` or at the root.

In [ ]:
# ============================================================
# Inspect the structure of an IMERG file
# ============================================================
# This step helps identify the groups, dimensions, and variables
# stored inside the NetCDF/HDF5 file.


def print_netcdf_structure(path, max_vars=15):
    """
    Print the groups, dimensions, and variables contained in
    a NetCDF/HDF5 file.
    """

    print(f"File: {os.path.basename(path)}")

    def _describe(group, indent=""):

        # Display the group name
        label = group.path if group.path != "/" else "/ (root)"
        print(f"{indent}Group: {label}")

        # Display dimensions
        if group.dimensions:
            dims = ", ".join(
                f"{name}={len(dim)}"
                for name, dim in group.dimensions.items()
            )
            print(f"{indent}  Dimensions: {dims}")

        # Display variables
        if group.variables:
            print(f"{indent}  Variables:")

            for i, (var_name, var) in enumerate(group.variables.items()):

                if i >= max_vars:
                    remaining = len(group.variables) - max_vars
                    print(f"{indent}    ... ({remaining} more)")
                    break

                print(
                    f"{indent}    - "
                    f"{var_name}{var.dimensions} ({var.dtype})"
                )

        # Repeat the same procedure for any nested groups
        for subgroup in group.groups.values():
            _describe(subgroup, indent + "  ")

    # Open the file and inspect its structure
    with netCDF4.Dataset(path) as ds:
        _describe(ds)


def get_imerg_group(path):
    """
    Return the name of the IMERG data group.

    Some IMERG products store their variables inside the 'Grid'
    group, while others store them directly in the root group.
    """

    with netCDF4.Dataset(path) as ds:

        if "Grid" in ds.groups:
            return "Grid"

        return None


# Inspect the first IMERG file used in the analysis
print_netcdf_structure(imerg_files[0])

### Load and combine the IMERG precipitation data

The IMERG files selected in the previous step are now opened with `xarray`.

For each file, the notebook:

1. automatically identifies whether the data are stored in the `Grid` group or in the root group;
2. reads the `precipitation` variable;
3. arranges the dimensions as `time, lat, lon`;
4. crops the data to the bounding box selected on the interactive map.

The individual files are then combined along the **time dimension** to create a single precipitation dataset covering the complete analysis period.

IMERG may also contain negative values used to represent missing or invalid observations. These values are masked before further processing.

In [ ]:
# ============================================================
# Load and combine IMERG precipitation data
# ============================================================

def load_imerg_stack(file_list, bbox):
    """
    Load multiple IMERG files, crop them to the selected area,
    and combine them into a single time series.
    Precipitation data organized as time, latitude, and longitude.
    """

    lon_min, lat_min, lon_max, lat_max = bbox

    datasets = []

    for file_path in file_list:

        # Detect whether the variables are stored in the "Grid" group or directly in the root group
        group = get_imerg_group(file_path)

        ds = xr.open_dataset(
            file_path,
            group=group,
            engine="netcdf4",
        )

        # Check that the precipitation variable is available
        if "precipitation" not in ds.variables:
            ds.close()
            raise KeyError(
                f"'precipitation' variable not found in:\n{file_path}"
            )

        da = ds["precipitation"]


        # Add a time dimension if it is not already present
        if "time" not in da.dims:
            da = da.expand_dims(
                time=[get_file_timestamp(file_path)]
            )


        # Reorder the spatial dimensions as time, lat, lon
        if "lat" in da.dims and "lon" in da.dims:
            da = da.transpose(..., "lat", "lon")


        # Crop the dataset to the selected study area
        da = da.sel(
            lon=slice(lon_min, lon_max),
            lat=slice(lat_min, lat_max),
        )


        # Load the selected data into memory before closing the file
        da = da.load()
        ds.close()

        datasets.append(da)


    # Combine all files along the time dimension
    precip = xr.concat(
        datasets,
        dim="time",
    ).sortby("time")


    # Mask negative values used for missing or invalid data
    precip = precip.where(precip >= 0)

    return precip


# Load the IMERG files selected in Section 2
precip = load_imerg_stack(
    imerg_files,
    bbox,
)

precip


### Check precipitation metadata

Before processing the precipitation data, the metadata of the first available IMERG file are inspected.

Metadata provide important information about how the precipitation values are stored and interpreted. In particular, this step allows us to check:

- the **units** of the `precipitation` variable;
- the numerical data type (`dtype`);
- any `scale_factor` or `add_offset` used to decode the stored values;
- the `_FillValue` used to represent missing or invalid data.

The precipitation units depend on the selected IMERG temporal resolution, so checking the metadata helps ensure that the data are interpreted correctly before further analysis.


In [ ]:
# ============================================================
# Check IMERG precipitation metadata
# ============================================================

# Use the first IMERG file as an example
sample_file = imerg_files[0]


# Detect whether the data are stored in the "Grid" group
# or directly in the root group
group = get_imerg_group(sample_file)


# Open the file
ds_check = xr.open_dataset(
    sample_file,
    group=group,
    engine="netcdf4",
)


# Select the precipitation variable
p_check = ds_check["precipitation"]

print("IMERG precipitation metadata")
print("----------------------------")
print(f"Temporal resolution: {time_scale}")
print(f"Units: {p_check.attrs.get('units', 'not specified')}")
print(f"dtype: {p_check.dtype}")
print(f"scale_factor: {p_check.encoding.get('scale_factor', 'not present')}")
print(f"add_offset: {p_check.encoding.get('add_offset', 'not present')}")
print(f"_FillValue: {p_check.encoding.get('_FillValue', 'not present')}")

ds_check.close()

### Convert precipitation to depth per timestep

IMERG products do not all express precipitation using the same temporal units. Before further analysis, the precipitation values are therefore converted to a common and more intuitive quantity: **precipitation depth per timestep**, expressed in millimetres (`mm`).

The conversion depends on the selected temporal resolution:

- **30-minute product:** precipitation is provided as a rate in `mm/hr`. Since each timestep represents 0.5 hours, the rate is multiplied by `0.5` to obtain precipitation depth in `mm` for each 30-minute interval.
- **Daily product:** precipitation is provided in `mm/day`. Because each timestep represents one full day, the values already correspond to daily precipitation depth and no conversion is required.
- **Monthly product:** precipitation is provided as a monthly mean precipitation rate in `mm/hr`. The rate is multiplied by `24` hours and by the number of days in the corresponding month to obtain the total monthly precipitation depth in `mm`.

The function below also checks the units stored in the IMERG metadata before applying the conversion. This helps prevent an incorrect conversion if an unexpected product or variable is loaded.


In [ ]:
# ============================================================
# Convert IMERG precipitation to depth per timestep
# ============================================================

def convert_to_precipitation_depth(precip, time_scale):
    """
    Convert IMERG precipitation to precipitation depth per timestep [mm].

    """

    # Read the precipitation units from the file metadata
    units = precip.attrs.get("units", "").strip().lower()


    # --------------------------------------------------------
    # 30-minute product
    # --------------------------------------------------------
    if time_scale == "30 min":

        if units not in ("mm/hr", "mm h-1", "mm/hour"):
            raise ValueError(
                f"Unexpected units for 30-minute IMERG data: '{units}'. "
                "Expected mm/hr."
            )

        # mm/hr × 0.5 hr = mm per 30-minute timestep
        precip_mm = precip * 0.5


    # --------------------------------------------------------
    # Daily product
    # --------------------------------------------------------
    elif time_scale == "Daily":

        if units not in ("mm/day", "mm d-1"):
            raise ValueError(
                f"Unexpected units for daily IMERG data: '{units}'. "
                "Expected mm/day."
            )

        # The value already represents precipitation depth accumulated over one day
        precip_mm = precip.copy()


    # --------------------------------------------------------
    # Monthly product
    # --------------------------------------------------------
    elif time_scale == "Monthly":

        if units not in ("mm/hr", "mm h-1", "mm/hour"):
            raise ValueError(
                f"Unexpected units for monthly IMERG data: '{units}'. "
                "Expected mm/hr."
            )

        # Number of days in each month
        days_in_month = precip["time"].dt.days_in_month

        # mm/hr × 24 hr/day × days/month = mm/month
        precip_mm = precip * 24 * days_in_month


    else:
        raise ValueError(
            f"Unsupported temporal resolution: '{time_scale}'"
        )


    # Update the metadata after the conversion
    precip_mm.attrs = precip.attrs.copy()
    precip_mm.attrs["units"] = "mm"
    precip_mm.attrs["long_name"] = (
        "Precipitation depth per timestep"
    )

    return precip_mm


# Convert the previously loaded IMERG data
precip = convert_to_precipitation_depth(
    precip,
    time_scale,
)

precip

### Precipitation maps over the selected area

For each IMERG grid cell, spatial maps are generated from the precipitation time series over the selected analysis period.

The following statistics are calculated:

- **Mean precipitation depth per timestep**
- **Maximum precipitation depth per timestep**

All values are expressed in `mm`. The meaning of a timestep depends on the selected IMERG product: 30 minutes, one day, or one month.

These statistics describe the spatial distribution of precipitation within the area selected in Section 1. The maps are generated only if the corresponding option was enabled in the analysis settings.


In [ ]:
# ============================================================
# Precipitation statistics for spatial mapping
# ============================================================
# Compute temporal statistics for each IMERG grid cell.

stat_map_mean = precip.mean(dim="time")
stat_map_max = precip.max(dim="time")


## 4. Precipitation maps over the selected area

This section is executed only if `run_area_maps = True`, according to the option selected in Section 1.

The maps are generated from the precipitation dataset previously loaded and converted to **precipitation depth per timestep** (`mm`). For each IMERG grid cell within the selected bounding box, the following statistics are displayed over the full analysis period:

- **Mean precipitation depth per timestep**
- **Maximum precipitation depth per timestep**

The meaning of a timestep depends on the selected IMERG product: **30 minutes**, **1 day**, or **1 month**.

If figure saving is enabled, each map is also saved to the output folder.


In [ ]:

# ============================================================
# Plot precipitation maps over the selected area
# ============================================================

def plot_stat_map(data2d, title, cmap, filename):
    """
    Plot a 2D precipitation map and optionally save it.
    """

    fig, ax = plt.subplots(figsize=(7, 6))

    data2d.plot(
        ax=ax,
        cmap=cmap,
        add_colorbar=True,
        cbar_kwargs={"label": "Precipitation (mm)"},
    )

    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    fig.tight_layout()

    maybe_save(fig, filename)
    plt.show()


if run_area_maps:

    plot_stat_map(
        stat_map_mean,
        f"Mean precipitation depth per timestep ({time_scale})",
        "Blues",
        "mean_precipitation_map.png",
    )

    plot_stat_map(
        stat_map_max,
        f"Maximum precipitation depth per timestep ({time_scale})",
        "YlGnBu",
        "maximum_precipitation_map.png",
    )

else:
    print("Precipitation maps are disabled.")


### Optional Excel export of grid-cell statistics

The spatial precipitation statistics can also be exported to an Excel file for further analysis.

Each row of the output table represents one **IMERG grid cell**, identified by its latitude and longitude. For each grid cell, the precipitation statistics calculated over the selected analysis period are exported:

- **Mean precipitation depth per timestep**
- **Maximum precipitation depth per timestep**

All precipitation values are expressed in `mm`. The meaning of a timestep depends on the selected IMERG product: 30 minutes, one day, or one month.

The Excel file contains two separate sheets (`mean`, and `max`) and is created only if the corresponding export option was enabled in Section 1.

In [ ]:
# ============================================================
# Export grid-cell precipitation statistics to Excel
# ============================================================

def stat_map_to_df(data2d, column_name):
    """
    Convert a 2D precipitation map into a tabular DataFrame.

    Each row represents one IMERG grid cell and contains
    its latitude, longitude, and precipitation statistic.
    """

    df = data2d.to_dataframe(
        name=column_name
    ).reset_index()

    return df[
        ["lat", "lon", column_name]
    ]


if run_area_excel:

    # Convert each precipitation map to a table
    df_mean = stat_map_to_df(
        stat_map_mean,
        "P_mean_mm",
    )

    df_max = stat_map_to_df(
        stat_map_max,
        "P_max_mm",
    )

    # Define the output Excel file
    excel_path = os.path.join(
        OUT_DIR,
        f"area_statistics_{time_scale.replace(' ', '_')}.xlsx",
    )


    # Save the three statistics in separate Excel sheets
    with pd.ExcelWriter(
        excel_path,
        engine="openpyxl",
    ) as writer:

        df_mean.to_excel(
            writer,
            sheet_name="mean",
            index=False,
        )

        df_max.to_excel(
            writer,
            sheet_name="max",
            index=False,
        )

    print(f"Excel file saved to: {excel_path}")

else:
    print("Grid-cell statistics Excel export is disabled.")


## 5. Loading the basin shapefile

The basin boundary is provided as an **ESRI Shapefile** and must be stored in the `shapefile/` folder.

A shapefile is composed of several associated files that must remain together in the same folder, typically including:

- `.shp` — geometry
- `.shx` — spatial index
- `.dbf` — attribute table
- `.prj` — coordinate reference system

The notebook checks that exactly one `.shp` file is available, loads the basin boundary, and reprojects it to **WGS84 (`EPSG:4326`)**, the geographic coordinate system used by the IMERG longitude and latitude grid.

The basin boundary will later be used to extract precipitation specifically over the catchment area.

In [ ]:
# ============================================================
# 5.1 Load and prepare the basin shapefile
# ============================================================

# Search for shapefiles in the shapefile folder
shp_files = glob.glob(
    os.path.join(SHP_DIR, "*.shp")
)


# Check that a shapefile is available
assert len(shp_files) > 0, (
    f"No .shp file was found in '{SHP_DIR}/'. "
    "Copy the basin shapefile and its associated files "
    "into this folder before continuing."
)


# This notebook expects only one basin shapefile
assert len(shp_files) == 1, (
    f"Found {len(shp_files)} .shp files in '{SHP_DIR}/'. "
    "The folder must contain only one basin shapefile."
)


# Path to the basin shapefile
shapefile_path = shp_files[0]


# Load the shapefile with GeoPandas
basin = gpd.read_file(shapefile_path)


# Check that the coordinate reference system is defined
assert basin.crs is not None, (
    "The shapefile has no coordinate reference system (CRS). "
    "Make sure that the associated .prj file is available."
)


# Reproject the basin to WGS84 to match the IMERG grid
basin = basin.to_crs("EPSG:4326")


# Display basic information
print(f"Shapefile loaded: {os.path.basename(shapefile_path)}")
print(f"Number of features: {len(basin)}")
print(f"Coordinate reference system: {basin.crs}")


### Shapefile validation

Before using the basin boundary, the shapefile is checked to ensure that it contains exactly **one feature** representing the catchment.

The geometry must also be a valid polygon (`Polygon` or `MultiPolygon`). If the shapefile contains multiple features, the workflow is stopped and the basin geometry must be prepared before continuing.

If the basin is divided into several features, they can be merged using a **dissolve** operation in GIS software. If the shapefile contains multiple basins, a new shapefile containing only the basin of interest should be created.

In [ ]:
# ============================================================
# 5.2 Validate the basin shapefile
# ============================================================

# The notebook expects exactly one feature representing the basin
if len(basin) != 1:

    print(
        f"WARNING: the shapefile contains {len(basin)} features "
        "instead of exactly one."
    )

    print(
        "This notebook requires a shapefile containing a SINGLE "
        "feature representing the basin."
    )

    print(
        " - If the basin is split into multiple features, merge them "
        "using a dissolve operation in GIS software."
    )

    print(
        " - If the shapefile contains multiple basins, create a new "
        "shapefile containing only the basin of interest."
    )

    raise ValueError(
        f"Invalid shapefile: found {len(basin)} features, "
        "but exactly one is required."
    )


# Extract the basin geometry
basin_geom = basin.geometry.iloc[0]


# Check that the geometry is not empty
if basin_geom is None or basin_geom.is_empty:
    raise ValueError(
        "The basin geometry is empty."
    )


# Check that the basin is represented by a polygon
if basin_geom.geom_type not in ("Polygon", "MultiPolygon"):
    raise ValueError(
        f"Invalid geometry type: {basin_geom.geom_type}. "
        "A Polygon or MultiPolygon is required."
    )


print("Shapefile validation successful.")
print(f"Geometry type: {basin_geom.geom_type}")

### Visual check of the basin boundary

The basin boundary is displayed on an interactive map to verify that the shapefile has been loaded and reprojected correctly.

The map automatically zooms to the basin extent.

In [ ]:
# ============================================================
# 5.3 Display the basin boundary
# ============================================================

# Get the geographic extent of the basin
lon_min, lat_min, lon_max, lat_max = basin.total_bounds


# Create an interactive map centered on the basin
basin_map = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(
        (lat_min + lat_max) / 2,
        (lon_min + lon_max) / 2,
    ),
    zoom=9,
)


# Add the basin boundary to the map
basin_layer = GeoJSON(
    data=basin.__geo_interface__,
    style={
        "color": "red",
        "weight": 3,
        "fillOpacity": 0.05,
    },
)

basin_map.add(basin_layer)


# Automatically zoom to the basin extent
basin_map.fit_bounds([
    [lat_min, lon_min],
    [lat_max, lon_max],
])


# Display the interactive map
basin_map

### Basin area calculation

The basin area cannot be calculated directly from geographic coordinates (`EPSG:4326`), because longitude and latitude are expressed in degrees rather than metres.

For this reason, the basin shapefile is temporarily reprojected to a local **UTM coordinate reference system**, automatically estimated from the basin location.

The basin area is then calculated in metric units and reported in both:

- square metres (`m²`)
- square kilometres (`km²`)

In [ ]:
# ============================================================
# Calculate basin area
# ============================================================

# Estimate an appropriate local UTM coordinate reference system
utm_crs = basin.estimate_utm_crs()

assert utm_crs is not None, (
    "A suitable UTM coordinate reference system could not be determined."
)


# Reproject the basin to the metric CRS
basin_utm = basin.to_crs(utm_crs)


# Calculate basin area in square metres
basin_area_m2 = float(
    basin_utm.geometry.iloc[0].area
)


# Convert the area to square kilometres
basin_area_km2 = basin_area_m2 / 1e6


# Display the results
print(f"Metric CRS used for area calculation: {utm_crs}")
print(f"Basin area: {basin_area_m2:,.0f} m²")
print(f"Basin area: {basin_area_km2:.2f} km²")


## 6. IMERG grid coverage of the basin

IMERG data are provided on a regular geographic grid defined by longitude and latitude coordinates (`EPSG:4326`). These coordinates represent the **centres of the IMERG grid cells**.

To compare the satellite grid with the basin boundary, the grid-cell boundaries are reconstructed from the longitude and latitude coordinates of the precipitation dataset.

The basin geometry and the IMERG grid must use the same coordinate reference system. The basin is therefore expressed in `EPSG:4326` before the spatial comparison.

This section checks whether the IMERG data loaded for the selected area fully cover the basin. If the basin extends outside the available IMERG grid, the workflow is stopped and a larger area of interest must be selected.

In [ ]:
# ============================================================
# 6.1 Check IMERG grid coverage of the basin
# ============================================================

def grid_cells_gdf(data2d, default_res=0.1):
    """
    Reconstruct IMERG grid-cell polygons from the longitude
    and latitude coordinates of a 2D DataArray.
    """

    lons = data2d["lon"].values
    lats = data2d["lat"].values


    # Estimate the grid spacing from the coordinate centres
    dlon = (
        np.abs(np.median(np.diff(lons)))
        if len(lons) > 1
        else default_res
    )

    dlat = (
        np.abs(np.median(np.diff(lats)))
        if len(lats) > 1
        else default_res
    )


    polygons = []
    lon_centers = []
    lat_centers = []


    # Reconstruct each grid cell around its centre coordinates
    for lat in lats:
        for lon in lons:

            polygons.append(
                box(
                    lon - dlon / 2,
                    lat - dlat / 2,
                    lon + dlon / 2,
                    lat + dlat / 2,
                )
            )

            lon_centers.append(lon)
            lat_centers.append(lat)


    return gpd.GeoDataFrame(
        {
            "lon": lon_centers,
            "lat": lat_centers,
        },
        geometry=polygons,
        crs="EPSG:4326",
    )


# Reconstruct the IMERG grid
grid_gdf = grid_cells_gdf(
    stat_map_mean
)


# Express the basin in the same CRS as the IMERG grid
basin_imerg = basin.to_crs(
    grid_gdf.crs
)


# Determine the geographic extent of the available IMERG grid
xmin, ymin, xmax, ymax = grid_gdf.total_bounds

grid_extent = box(
    xmin,
    ymin,
    xmax,
    ymax,
)


# Basin geometry in the IMERG coordinate system
basin_geom_imerg = basin_imerg.geometry.iloc[0]


# Check whether the available grid fully covers the basin
if grid_extent.covers(basin_geom_imerg):

    print("IMERG spatial coverage check: OK")
    print("The available IMERG grid fully covers the basin.")

else:

    raise ValueError(
        "The available IMERG grid does not fully cover the basin. "
        "Select a larger area of interest and load the IMERG data again."
    )

### Visual inspection of the IMERG grid

The reconstructed IMERG grid and the basin boundary are displayed together for a visual check.

The grey lines represent the individual IMERG grid cells, while the red line represents the basin boundary.

In [ ]:
# ============================================================
# 6.2 Display the IMERG grid and basin boundary
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 7)
)


# Plot IMERG grid-cell boundaries
grid_gdf.boundary.plot(
    ax=ax,
    linewidth=0.4,
    color="grey",
)


# Plot the basin boundary
basin_imerg.boundary.plot(
    ax=ax,
    color="red",
    linewidth=1.5,
)


ax.set_title("IMERG grid and basin boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

fig.tight_layout()


# Save the figure if this option was enabled
maybe_save(
    fig,
    "imerg_grid_vs_basin.png",
)

plt.show()

## 7. Precipitation maps over the basin

This section displays the precipitation statistics previously computed from the IMERG dataset, now zoomed to the spatial extent of the basin.

The basin boundary is overlaid on each map to facilitate visual comparison between the catchment shape and the surrounding IMERG grid cells.

The following maps are shown:

- **Mean precipitation depth per timestep**
- **Maximum precipitation depth per timestep**

All values are expressed in `mm`. The meaning of a timestep depends on the selected IMERG product: 30 minutes, one day, or one month.

Note that these maps are **not clipped to the basin boundary**. The full IMERG grid is still displayed, but the map extent is restricted to the basin area.

In [ ]:
# ============================================================
# Plot precipitation maps zoomed to the basin
# ============================================================

def plot_stat_map_zoom(data2d, title, cmap, filename, basin_gdf):
    """
    Plot a precipitation map zoomed to the basin extent,
    with the basin boundary overlaid.
    """

    # Get basin extent
    minx, miny, maxx, maxy = basin_gdf.total_bounds

    # Add a small margin around the basin for better visualization
    pad_x = (maxx - minx) * 0.15
    pad_y = (maxy - miny) * 0.15

    fig, ax = plt.subplots(figsize=(7, 6))

    # Plot the precipitation statistic
    data2d.plot(
        ax=ax,
        cmap=cmap,
        add_colorbar=True,
        cbar_kwargs={"label": "Precipitation (mm)"},
    )

    # Overlay the basin boundary
    basin_gdf.boundary.plot(
        ax=ax,
        color="red",
        linewidth=1.5,
    )

    # Zoom to the basin extent
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    fig.tight_layout()

    maybe_save(fig, filename)
    plt.show()


plot_stat_map_zoom(
    stat_map_mean,
    f"Mean precipitation depth per timestep ({time_scale}) - basin view",
    "Blues",
    "basin_mean_precipitation_map.png",
    basin,
)

plot_stat_map_zoom(
    stat_map_max,
    f"Maximum precipitation depth per timestep ({time_scale}) - basin view",
    "YlGnBu",
    "basin_maximum_precipitation_map.png",
    basin,
)

## 8. Basin-averaged precipitation — three methods

Basin-averaged precipitation is estimated using three different spatial aggregation approaches:

1. **`all_touched=True` method**  
   Basin precipitation is computed from the IMERG grid cells selected by rasterizing the basin polygon. All grid cells touched by the basin boundary are included.

2. **`all_touched=False` method** 
   Basin precipitation is computed from the IMERG grid cells selected by rasterizing the basin polygon using the default rasterization rule.

   
3. **Area-weighted grid-cell method**  
   Each IMERG grid cell intersecting the basin is weighted according to the fraction of its area that lies inside the basin. Basin-averaged precipitation is then calculated as the weighted mean of the contributing grid cells.

All three methods are computed for comparison. Their resulting precipitation time series are compared in the following section, and one method is then selected for the final export.


### `all_touched` methods -  True/False

The first two methods are based on **rasterizing** the basin polygon onto the IMERG grid.

This means that the vector basin boundary is converted into a boolean mask with the same spatial structure as the IMERG grid. Each grid cell is classified as either:

- **inside the basin** (`True`)
- **outside the basin** (`False`)

Two rasterization options are compared:

- **`all_touched=False`**: uses the default rasterization rule;
- **`all_touched=True`**: includes all grid cells touched by the basin boundary.

The resulting masks are then used to compute basin-averaged precipitation time series from the IMERG dataset.

In [ ]:
# ============================================================
# 8.1 Basin masks based on rasterization
# ============================================================

def rasterize_basin_mask(data2d, basin_geometry, all_touched):
    """
    Rasterize the basin geometry onto the IMERG grid.
    """

    lons = data2d["lon"].values
    lats = data2d["lat"].values

    # Estimate grid spacing from IMERG coordinates
    dlon = np.abs(np.median(np.diff(lons)))
    dlat = np.abs(np.median(np.diff(lats)))

    # Rasterio expects a north-up grid, so latitudes must be
    # ordered from north to south during rasterization
    lat_desc = lats[::-1] if lats[0] < lats[-1] else lats

    # Build the affine transform of the IMERG grid
    transform = from_origin(
        lons.min() - dlon / 2,
        lat_desc.max() + dlat / 2,
        dlon,
        dlat,
    )

    out_shape = (len(lat_desc), len(lons))

    # Rasterize the basin geometry
    mask = features.geometry_mask(
        [basin_geometry],
        out_shape=out_shape,
        transform=transform,
        invert=True,
        all_touched=all_touched,
    )

    # Convert the mask to an xarray DataArray
    mask_da = xr.DataArray(
        mask,
        dims=["lat", "lon"],
        coords={
            "lat": lat_desc,
            "lon": lons,
        },
    )

    # Reorder latitude to match the original IMERG data
    mask_da = mask_da.sortby("lat")

    return mask_da


# Create basin masks using the two rasterization options
mask_touched_true = rasterize_basin_mask(
    stat_map_mean,
    basin_geom,
    all_touched=True,
)

mask_touched_false = rasterize_basin_mask(
    stat_map_mean,
    basin_geom,
    all_touched=False,
)


# Compute basin-averaged precipitation time series
areal_touched_true = precip.where(
    mask_touched_true
).mean(dim=["lat", "lon"])

areal_touched_false = precip.where(
    mask_touched_false
).mean(dim=["lat", "lon"])


# Display the number of selected IMERG grid cells
print(
    "Selected grid cells (all_touched=True):",
    int(mask_touched_true.sum()),
)

print(
    "Selected grid cells (all_touched=False):",
    int(mask_touched_false.sum()),
)

### Visual comparison of the rasterization masks

The two rasterization masks are displayed on the IMERG grid for visual inspection.

In each panel:

- the **grey lines** represent the IMERG grid;
- the **black points** represent the grid-cell centres;
- the **blue cells** are the grid cells selected by the rasterization mask;
- the **red outline** represents the basin boundary.

This comparison helps illustrate how the choice of `all_touched=True` or `all_touched=False` affects which IMERG grid cells are included in the basin average.

In [ ]:
# ============================================================
# 8.2 Visual comparison of rasterization masks
# ============================================================

from matplotlib.patches import Rectangle


def plot_rasterization_masks(data2d, basin_geometry, mask_false, mask_true):
    """
    Visualize the effect of all_touched=False and all_touched=True on the IMERG grid.
    """

    lons = data2d["lon"].values
    lats = data2d["lat"].values

    dlon = np.abs(np.median(np.diff(lons)))
    dlat = np.abs(np.median(np.diff(lats)))

    # Reconstruct grid-cell edges from grid-cell centres
    lon_edges = np.concatenate([
        [lons[0] - dlon / 2],
        lons + dlon / 2,
    ])

    lat_edges = np.concatenate([
        [lats[0] - dlat / 2],
        lats + dlat / 2,
    ])

    # Basin boundary as a GeoSeries for plotting
    basin_gs = gpd.GeoSeries(
        [basin_geometry],
        crs="EPSG:4326",
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 6),
        sharex=True,
        sharey=True,
    )

    masks = [
        (mask_false, "all_touched=False"),
        (mask_true, "all_touched=True"),
    ]

    for ax, (mask, title) in zip(axes, masks):

        # Show selected grid cells
        selected = np.where(
            mask.values,
            1.0,
            np.nan,
        )

        ax.pcolormesh(
            lon_edges,
            lat_edges,
            selected,
            cmap="Blues",
            alpha=0.5,
            shading="flat",
        )

        # Add a border around selected grid cells
        for i, lat in enumerate(lats):
            for j, lon in enumerate(lons):

                if mask.values[i, j]:
                    rect = Rectangle(
                        (lon - dlon / 2, lat - dlat / 2),
                        dlon,
                        dlat,
                        facecolor="none",
                        edgecolor="navy",
                        linewidth=1.2,
                        zorder=3,
                    )
                    ax.add_patch(rect)

        # Draw IMERG grid lines
        for lon_edge in lon_edges:
            ax.axvline(
                lon_edge,
                color="grey",
                linewidth=0.5,
                alpha=0.7,
            )

        for lat_edge in lat_edges:
            ax.axhline(
                lat_edge,
                color="grey",
                linewidth=0.5,
                alpha=0.7,
            )

        # Plot grid-cell centres
        lon_grid, lat_grid = np.meshgrid(lons, lats)

        ax.scatter(
            lon_grid,
            lat_grid,
            s=12,
            color="black",
            zorder=4,
            label="Grid-cell centres",
        )

        # Plot basin boundary
        basin_gs.boundary.plot(
            ax=ax,
            color="red",
            linewidth=2,
            zorder=5,
        )

        ax.set_title(title)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_aspect("equal")

    fig.tight_layout()

    maybe_save(
        fig,
        "all_touched_comparison.png",
    )

    plt.show()


plot_rasterization_masks(
    stat_map_mean,
    basin_geom,
    mask_touched_false,
    mask_touched_true,
)

### Area-weighted grid-cell method

The third method accounts explicitly for the fraction of each IMERG grid cell that overlaps the basin.

Unlike the two rasterization methods, which classify each grid cell as either included or excluded, the area-weighted approach assigns a different contribution to each intersecting cell according to the area that actually falls inside the catchment.

The procedure consists of the following steps:

1. reconstruct the IMERG grid-cell polygons from the longitude and latitude coordinates;
2. identify the grid cells that intersect the basin;
3. reproject the basin and grid cells to a local metric coordinate reference system, so that areas can be calculated in square metres;
4. calculate the intersection area between each IMERG cell and the basin;
5. convert the intersection areas into normalized spatial weights;
6. use these weights to calculate the basin-averaged precipitation for each timestep.

Two quantities are calculated for each contributing grid cell:

- **`fraction_cell_inside`**: fraction of the individual IMERG grid cell that lies inside the basin;
- **`weight`**: fraction of the total basin-covered IMERG area represented by that cell. These normalized weights sum to 1.

If precipitation is missing for one or more grid cells at a given timestep, the available weights are automatically renormalized so that the basin average is calculated only from valid observations.

In [ ]:
# ============================================================
# 8.3 Area-weighted basin precipitation
# ============================================================

def compute_area_weighted_precipitation(data2d, precip, basin_gdf):
    """
    Compute basin-averaged precipitation by weighting each IMERG
    grid cell according to the area that overlaps the basin.

    Returns
    -------
    areal_weighted : xarray.DataArray
        Area-weighted basin precipitation time series.
    weights_da : xarray.DataArray
        Two-dimensional array containing the normalized spatial
        weights assigned to the IMERG grid cells.
    grid_metric : geopandas.GeoDataFrame
        Table containing the contributing IMERG cells and their
        areas, intersection fractions, and normalized weights.
    intersection_gdf : geopandas.GeoDataFrame
        Portions of the IMERG grid cells falling inside the basin.
    """

    # --------------------------------------------------------
    # 1. Reconstruct the IMERG grid
    # --------------------------------------------------------

    grid_gdf = grid_cells_gdf(data2d)

    # Express the basin in the same CRS as the IMERG grid
    basin_grid_crs = basin_gdf.to_crs(grid_gdf.crs)
    basin_geom_grid = basin_grid_crs.geometry.iloc[0]

    # Keep only grid cells that intersect the basin
    grid_intersecting = grid_gdf[
        grid_gdf.intersects(basin_geom_grid)
    ].copy()

    if grid_intersecting.empty:
        raise ValueError(
            "No IMERG grid cells intersect the basin."
        )


    # --------------------------------------------------------
    # 2. Reproject to a metric CRS
    # --------------------------------------------------------
    # Areas must be calculated in a projected coordinate system.

    metric_crs = basin_grid_crs.estimate_utm_crs()

    if metric_crs is None:
        raise ValueError(
            "A suitable metric coordinate reference system "
            "could not be determined."
        )

    basin_metric = basin_grid_crs.to_crs(metric_crs)
    grid_metric = grid_intersecting.to_crs(metric_crs)

    basin_geom_metric = basin_metric.geometry.iloc[0]


    # --------------------------------------------------------
    # 3. Calculate grid-cell and intersection areas
    # --------------------------------------------------------

    # Total area of each IMERG grid cell
    grid_metric["cell_area_m2"] = (
        grid_metric.geometry.area
    )

    # Portion of each cell that falls inside the basin
    grid_metric["intersection_geom"] = (
        grid_metric.geometry.intersection(
            basin_geom_metric
        )
    )

    grid_metric["intersection_area_m2"] = (
        grid_metric["intersection_geom"].area
    )

    # Remove cells with no actual overlap
    grid_metric = grid_metric[
        grid_metric["intersection_area_m2"] > 0
    ].copy()


    # Fraction of each individual grid cell inside the basin
    grid_metric["fraction_cell_inside"] = (
        grid_metric["intersection_area_m2"]
        / grid_metric["cell_area_m2"]
    )


    # --------------------------------------------------------
    # 4. Calculate normalized spatial weights
    # --------------------------------------------------------

    total_intersection_area = (
        grid_metric["intersection_area_m2"].sum()
    )

    basin_area_m2 = basin_geom_metric.area

    coverage_fraction = (
        total_intersection_area / basin_area_m2
    )


    print(
        f"Basin area: "
        f"{basin_area_m2 / 1e6:.2f} km²"
    )

    print(
        f"IMERG grid area inside the basin: "
        f"{total_intersection_area / 1e6:.2f} km²"
    )

    print(
        f"Basin coverage: "
        f"{coverage_fraction * 100:.2f}%"
    )


    # Normalize the intersection areas so that weights sum to 1
    grid_metric["weight"] = (
        grid_metric["intersection_area_m2"]
        / total_intersection_area
    )


    # --------------------------------------------------------
    # 5. Build the intersection GeoDataFrame
    # --------------------------------------------------------

    intersection_gdf = gpd.GeoDataFrame(
        grid_metric[
            [
                "lon",
                "lat",
                "cell_area_m2",
                "intersection_area_m2",
                "fraction_cell_inside",
                "weight",
            ]
        ].copy(),
        geometry=grid_metric["intersection_geom"],
        crs=metric_crs,
    )


    # --------------------------------------------------------
    # 6. Build a weight array aligned with the IMERG grid
    # --------------------------------------------------------

    weights_da = xr.zeros_like(
        data2d,
        dtype=float,
    )

    for _, row in grid_metric.iterrows():

        weights_da.loc[
            dict(
                lat=row["lat"],
                lon=row["lon"],
            )
        ] = row["weight"]


    # --------------------------------------------------------
    # 7. Calculate area-weighted precipitation
    # --------------------------------------------------------

    # Ignore weights where precipitation data are missing
    valid_weights = weights_da.where(
        precip.notnull()
    )

    # Weighted mean for each timestep
    areal_weighted = (
        (precip * valid_weights).sum(
            dim=["lat", "lon"]
        )
        / valid_weights.sum(
            dim=["lat", "lon"]
        )
    )


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print(
        "IMERG cells contributing to the area-weighted mean:",
        len(grid_metric),
    )

    print(
        "Sum of normalized weights:",
        f"{grid_metric['weight'].sum():.6f}",
    )


    return (
        areal_weighted,
        weights_da,
        grid_metric,
        intersection_gdf,
    )


# Calculate the area-weighted basin precipitation
(
    areal_weighted,
    area_weights,
    weights_table,
    intersection_gdf,
) = compute_area_weighted_precipitation(
    stat_map_mean,
    precip,
    basin,
)


# Display the spatial weights assigned to each IMERG cell
weights_table[
    [
        "lon",
        "lat",
        "cell_area_m2",
        "intersection_area_m2",
        "fraction_cell_inside",
        "weight",
    ]
]

### Visualizing the area-weighted grid cells

The area-weighted method can be visualized by showing the portions of the IMERG grid cells that actually fall inside the basin.

In the figure:

- the **grey lines** represent the full IMERG grid cells intersecting the basin;
- the **colored areas** represent the portions of those cells located inside the basin;
- the color intensity represents the **normalized weight** assigned to each grid cell;
- the **red line** represents the basin boundary.

Grid cells contributing a larger fraction of the basin area receive a larger weight in the basin-averaged precipitation calculation.

In [ ]:
# ============================================================
# 8.4 Visualize area-weighted IMERG grid cells
# ============================================================

def plot_area_weights(weights_table, intersection_gdf, basin_gdf):
    """
    Visualize the IMERG grid cells contributing to the
    area-weighted basin precipitation.
    """

    # Reproject all geometries to WGS84 for visualization
    full_cells = weights_table.to_crs("EPSG:4326")
    intersections = intersection_gdf.to_crs("EPSG:4326")
    basin_plot = basin_gdf.to_crs("EPSG:4326")


    fig, ax = plt.subplots(
        figsize=(8, 7)
    )


    # Plot the full IMERG grid cells
    full_cells.boundary.plot(
        ax=ax,
        color="grey",
        linewidth=0.8,
        alpha=0.8,
    )


    # Plot the portions of the cells falling inside the basin,
    # colored according to their normalized weight
    intersections.plot(
        ax=ax,
        column="weight",
        cmap="Blues",
        edgecolor="navy",
        linewidth=0.8,
        legend=True,
        legend_kwds={
            "label": "Normalized weight",
        },
    )


    # Overlay the basin boundary
    basin_plot.boundary.plot(
        ax=ax,
        color="red",
        linewidth=2,
    )


    # Zoom to the basin area
    minx, miny, maxx, maxy = basin_plot.total_bounds

    pad_x = (maxx - minx) * 0.15
    pad_y = (maxy - miny) * 0.15

    ax.set_xlim(
        minx - pad_x,
        maxx + pad_x,
    )

    ax.set_ylim(
        miny - pad_y,
        maxy + pad_y,
    )


    ax.set_title(
        "Area-weighted IMERG grid cells"
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    fig.tight_layout()

    maybe_save(
        fig,
        "area_weighted_grid_cells.png",
    )

    plt.show()


plot_area_weights(
    weights_table,
    intersection_gdf,
    basin,
)

## 9. Comparison of basin-averaged precipitation methods

This section compares the basin-averaged precipitation time series obtained with the three spatial aggregation methods introduced in Section 8:

- **`all_touched=False`**
- **`all_touched=True`**
- **Area-weighted**

The purpose is to evaluate how the choice of spatial aggregation method affects the resulting basin precipitation estimates.

### Time series — comparison of precipitation values

The following plot shows the basin-averaged precipitation time series computed with each method over the selected analysis period.

All values are expressed in `mm` per timestep. The meaning of a timestep depends on the selected IMERG product: 30 minutes, one day, or one month.

In [ ]:
# ============================================================
# 9.1 Compare basin-averaged precipitation time series
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5))


# Plot the basin-averaged precipitation time series
areal_touched_false.plot(
    ax=ax,
    label="all_touched=False",
    marker="o",
    markersize=3,
)

areal_touched_true.plot(
    ax=ax,
    label="all_touched=True",
    marker="o",
    markersize=3,
)

areal_weighted.plot(
    ax=ax,
    label="Area-weighted",
    marker="o",
    markersize=3,
)


ax.set_ylabel("Basin-averaged precipitation (mm)")
ax.set_xlabel("Date")

ax.set_title(
    f"Basin-averaged precipitation ({time_scale}) - method comparison"
)

ax.legend()

fig.tight_layout()

maybe_save(
    fig,
    "basin_averaged_precipitation_method_comparison.png",
)

plt.show()


### Cumulative precipitation comparison

The cumulative basin-averaged precipitation is calculated for each spatial aggregation method by progressively summing the precipitation depth over time.

Comparing the cumulative series makes it easier to identify whether small differences among the methods accumulate over the analysis period.

Three situations are handled separately:

- if a method contains valid precipitation values for the entire period, the cumulative series is calculated normally;
- if only some timesteps are missing, the available values are still plotted and a warning reports the dates containing missing data;
- if the entire time series contains `NaN` values, no IMERG grid cells were selected by that method and the method is considered not applicable.

For partially incomplete series, missing timesteps are not interpreted as zero precipitation. They remain `NaN` in the plotted series, producing visible gaps in the curve. Cumulative values after a missing timestep therefore represent precipitation accumulated from the available observations only.

The final value of each curve represents the total basin-averaged precipitation accumulated over the selected analysis period.


In [ ]:
# ============================================================
# 9.2 Compare cumulative basin-averaged precipitation
# ============================================================

def cumulative_precipitation(series, method_name):
    """
    Calculate cumulative precipitation while identifying
    unavailable methods and missing timesteps.

    Returns
    -------
    cumulative : xarray.DataArray
        Cumulative precipitation series.
    missing_dates : list
        Dates/times where precipitation values are missing.
    """

    # --------------------------------------------------------
    # Case 1: the entire series is NaN
    # --------------------------------------------------------
    if bool(series.isnull().all()):

        print(
            f"{method_name}: no IMERG grid cells were selected. "
            "This method is not applicable."
        )

        return series.copy(), []


    # --------------------------------------------------------
    # Case 2: only some timesteps are missing
    # --------------------------------------------------------
    missing_mask = series.isnull()

    missing_dates = pd.to_datetime(
        series["time"].values[missing_mask.values]
    ).tolist()


    if missing_dates:

        print(
            f"WARNING - {method_name}: "
            f"{len(missing_dates)} missing timestep(s) found."
        )

        print("Missing dates/times:")

        for date in missing_dates:
            print(f"  - {date}")


    # --------------------------------------------------------
    # Calculate cumulative precipitation from available values
    # --------------------------------------------------------
    # Missing values are temporarily ignored for the cumulative
    # calculation, but are restored as NaN in the plotted series.

    cumulative = (
        series.fillna(0)
        .cumsum(dim="time")
        .where(series.notnull())
    )

    return cumulative, missing_dates


# Calculate cumulative precipitation
cumulative_touched_false, missing_false = cumulative_precipitation(
    areal_touched_false,
    "all_touched=False",
)

cumulative_touched_true, missing_true = cumulative_precipitation(
    areal_touched_true,
    "all_touched=True",
)

cumulative_weighted, missing_weighted = cumulative_precipitation(
    areal_weighted,
    "Area-weighted",
)


# ============================================================
# Plot cumulative precipitation
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5))


# Plot only methods containing at least one valid value
if not bool(cumulative_touched_false.isnull().all()):
    cumulative_touched_false.plot(
        ax=ax,
        label="all_touched=False",
    )

if not bool(cumulative_touched_true.isnull().all()):
    cumulative_touched_true.plot(
        ax=ax,
        label="all_touched=True",
    )

if not bool(cumulative_weighted.isnull().all()):
    cumulative_weighted.plot(
        ax=ax,
        label="Area-weighted",
    )


ax.set_ylabel("Cumulative basin-averaged precipitation (mm)")
ax.set_xlabel("Date")

ax.set_title(
    f"Cumulative basin-averaged precipitation ({time_scale}) "
    "- method comparison"
)

ax.legend()

fig.tight_layout()

maybe_save(
    fig,
    "cumulative_basin_precipitation_method_comparison.png",
)

plt.show()


## 10. Select the final method

One of the basin-averaging methods calculated in Section 8 can now be selected to generate the final basin precipitation time series.

The selected method will be used for:

- the final basin-averaged precipitation plot;
- the final Excel export.

Only methods that contain at least one valid precipitation value are made available for selection. A method that produced only `NaN` values, for example because no IMERG grid cells were selected, is considered not applicable and is excluded.

In [ ]:
# ============================================================
# 10.1 Select the final basin-averaging method
# ============================================================

# Store the precipitation time series produced by each method
all_method_series = {
    "all_touched=True": areal_touched_true,
    "all_touched=False": areal_touched_false,
    "Area-weighted": areal_weighted,
}


# Keep only methods containing at least one valid value
method_series = {
    name: series
    for name, series in all_method_series.items()
    if not bool(series.isnull().all())
}


# Report methods that are not applicable
excluded_methods = [
    name
    for name, series in all_method_series.items()
    if bool(series.isnull().all())
]

if excluded_methods:
    print(
        "Methods excluded because no valid basin precipitation "
        "values were available:"
    )

    for name in excluded_methods:
        print(f"  - {name}")


# Make sure that at least one method is available
assert len(method_series) > 0, (
    "No valid basin-averaging method is available."
)


# Use the area-weighted method as the default when available
default_method = (
    "Area-weighted"
    if "Area-weighted" in method_series
    else list(method_series.keys())[0]
)


# Create the interactive method selector
method_w = widgets.Dropdown(
    options=list(method_series.keys()),
    value=default_method,
    description="Basin-averaging method:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px"),
)


display(method_w)


### Basin-averaged precipitation time series

After selecting the basin-averaging method, the corresponding precipitation time series is used as the final basin precipitation dataset.

The selected series is plotted over the analysis period and can optionally be saved as a figure.

If some timesteps contain missing values (`NaN`), the available precipitation values are still displayed and the missing dates are reported. Missing values are not replaced by zero.

In [ ]:
# ============================================================
# 10.2 Plot the selected basin-averaged precipitation series
# ============================================================
# Run this cell AFTER selecting the method from the dropdown above.


# Retrieve the selected method and its precipitation time series
selected_method_name = method_w.value
selected_series = method_series[selected_method_name]

print(f"Selected method: {selected_method_name}")


# ------------------------------------------------------------
# Check for missing values
# ------------------------------------------------------------

missing_mask = selected_series.isnull()

missing_dates = pd.to_datetime(
    selected_series["time"].values[missing_mask.values]
).tolist()


if missing_dates:

    print(
        f"WARNING: {len(missing_dates)} missing timestep(s) "
        "found in the selected series."
    )

    print("Missing dates/times:")

    for date in missing_dates:
        print(f"  - {date}")


# ------------------------------------------------------------
# Plot the selected precipitation time series
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 5))

selected_series.plot(
    ax=ax,
    marker="o",
    markersize=3,
    color="#1d3557",
)


# ------------------------------------------------------------
# Format the time axis automatically
# ------------------------------------------------------------

# Automatically choose an appropriate spacing between date ticks
date_locator = mdates.AutoDateLocator(
    minticks=3,
    maxticks=10,
)

ax.xaxis.set_major_locator(date_locator)


# Determine the duration of the selected time series
time_values = pd.to_datetime(selected_series["time"].values)

time_span_days = (
    time_values.max() - time_values.min()
).total_seconds() / 86400


# Choose a numeric date format according to the time span
if time_span_days <= 2:
    # Short periods: show date and time
    date_format = "%d/%m/%Y\n%H:%M"

elif time_span_days <= 120:
    # Periods of a few days or months
    date_format = "%d/%m/%Y"

elif time_span_days <= 730:
    # Periods up to about two years
    date_format = "%m/%Y"

else:
    # Long multi-year periods
    date_format = "%Y"


ax.xaxis.set_major_formatter(
    mdates.DateFormatter(date_format)
)

# Rotate date labels to avoid overlapping
plt.setp(
    ax.get_xticklabels(),
    rotation=30,
    ha="right",
)

# Add a small horizontal margin at both ends of the plot
ax.margins(x=0.02)
# ------------------------------------------------------------
# Figure labels
# ------------------------------------------------------------

ax.set_ylabel("Basin-averaged precipitation (mm)")
ax.set_xlabel("Date")

ax.set_title(
    f"Basin-averaged precipitation ({time_scale}) "
    f"- method: {selected_method_name}"
)

ax.grid(
    axis="x",
    alpha=0.2,
)

fig.tight_layout()


# ------------------------------------------------------------
# Create a filename-safe version of the method name
# ------------------------------------------------------------

safe_method_name = (
    selected_method_name
    .replace(" ", "_")
    .replace("(", "")
    .replace(")", "")
    .replace("=", "")
)


# Save the figure if this option was enabled
maybe_save(
    fig,
    f"basin_averaged_precipitation_{safe_method_name}.png",
)

plt.show()

### Export the basin-averaged precipitation series

The precipitation time series obtained with the selected basin-averaging method is exported to an Excel file.

The output table contains:

- `date` — date or timestamp of the precipitation observation;
- `basin_averaged_precipitation_mm` — precipitation depth averaged over the basin, expressed in `mm` per timestep.

The meaning of a timestep depends on the selected IMERG product: 30 minutes, one day, or one month.

Any missing precipitation values (`NaN`) are preserved in the exported file and are not replaced by zero.

In [ ]:
# ============================================================
# 10.3 Export the final basin precipitation series to Excel
# ============================================================

def to_series_df(da):
    """
    Convert a basin-averaged precipitation time series
    to a DataFrame ready for Excel export.
    """

    # Convert the xarray time series to a table
    df = da.to_dataframe(
        name="precipitation_mm"
    ).reset_index()

    # Keep only time and precipitation
    df = df[
        ["time", "precipitation_mm"]
    ]

    # Use clear column names for the exported file
    df.columns = [
        "date",
        "basin_averaged_precipitation_mm",
    ]

    return df


# Convert the selected precipitation series to a DataFrame
df_areal = to_series_df(
    selected_series
)


# Define the output Excel file
excel_areal_path = os.path.join(
    OUT_DIR,
    (
        f"basin_averaged_precipitation_"
        f"{time_scale.replace(' ', '_')}_"
        f"{safe_method_name}.xlsx"
    ),
)


# Export the precipitation series
df_areal.to_excel(
    excel_areal_path,
    index=False,
)


print(f"Excel file saved to: {excel_areal_path}")


## Contacts

For questions, comments, or further information about this notebook, please contact:

**Sofia Ortenzi**  
CNR-GEO — Institute of Geosciences  
Perugia, Italy  
📧 *[sofia.ortenzi@cnr.it]*

**Lucio Di Matteo**  
University of Perugia, Department of Physics and Geology 
Perugia, Italy  
📧 *[lucio.dimatteo@unipg.it]*


